# 04f_draftsharks_two_tree

**Purpose:** pull BOTH DraftSharks ranking trees — the **dynasty** board and
the **redraft** board — into `fact_dynasty_ranking_metrics`.

**Why both.** `mouserat_trade-bud`'s stance selector picks a ranking *source*,
not a tuned weight: Contending reads the redraft ordering, Future/Balanced read
the dynasty ordering, for offense and defense alike. The redraft ordering is
expert-produced and is **not** derivable from the dynasty columns — the two
boards disagree from the very top (dynasty: Allen, Maye, Daniels; redraft:
Allen, Nacua, Chase). So it has to be pulled, not computed.

**Metric keys.** Both trees are the same vendor, so both land under
`source_name = "DynastySharks"`. They cannot share the `ds_` prefix without
breaking the invariant that one `metric_key` maps to exactly one `source_name`
— the functional dependency that lets `source_name` live on
`dim_dynasty_metric` instead of on the fact. The redraft tree therefore uses
`dsr_*`, via `etl.fold_ranks_long(prefix="dsr")`. Rows, never columns.

**Endpoint.** `load-rows`, **not** `load-table` (which caps at 250 regardless
of `limit`). `/export` is paywalled.

```
GET https://www.draftsharks.com/{base}/load-rows
    ?offset=N&limit=250&pprSuperflexSlug=te-premium-superflex&playerGroup=all&sort=
headers: Referer=https://www.draftsharks.com/{base}/te-premium-superflex, HX-Request: true
```

**Parsing.** Split on `<tbody`, keep blocks containing `data-player-row`, then
run a **per-attribute** regex on each block. A single combined regex silently
returns zero rows: the `data-*` attributes are newline-separated in the markup,
so `[^>]*` between them never matches.

**Identity.** `source_player_id` is a locally-minted name slug, identical to the
rule `04x_manual_dynasty_rankings` already uses, so these rows land in the SAME
ID namespace as the existing DynastySharks crosswalk entries. The vendor's
numeric `data-key` is deliberately *not* the ID — adopting it would strand the
existing resolved rows in a second namespace.

**Supersedes:** the DynastySharks sheets in `04x_manual_dynasty_rankings`
(which keeps FantasyPros).

**Output:** rows appended to `data/fact_dynasty_ranking_metrics.parquet`
(replace-by-`(snapshot_date, source_name)`) + crosswalk upserts into
`data/dim_dynasty_crosswalk.parquet`.

In [1]:
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA, REVIEW

import json
import re
import time

import pandas as pd

SOURCE = "DynastySharks"
FORMAT = "TEPP"   # the league's scoring: TE-premium superflex
SLUG = "te-premium-superflex"
PAGE = 250        # server-side max per load-rows call

# base path -> metric_key prefix. Both are DynastySharks; the prefix is what
# keeps their metric_keys distinct (see dim_dynasty_metric / 04c).
TREES = {"dynasty-rankings": "ds", "rankings": "dsr"}

# A tree that suddenly returns far fewer rows means the markup or the endpoint
# changed; fail rather than silently shrinking a ranking source.
MIN_ROWS = 700

In [2]:
# ---- Extract ----------------------------------------------------------------
_ATTRS = {"vendor_key": "data-key",
          "player_name": "data-player-name",
          "position_raw": "data-fantasy-position"}


def parse_rows(html: str) -> list[dict]:
    """Rows out of a load-rows fragment.

    Per-attribute regex over per-row blocks, NOT one combined pattern: the
    data-* attributes are newline-separated in DraftSharks' markup, so a
    combined `data-key=".."[^>]*data-player-name=".."` matches nothing and
    returns an empty board with a 200 OK.
    """
    out = []
    for block in html.split("<tbody"):
        if "data-player-row" not in block:
            continue
        row = {k: (m.group(1) if (m := re.search(rf'{a}="([^"]+)"', block)) else None)
               for k, a in _ATTRS.items()}
        if row["player_name"]:
            out.append(row)
    return out


def fetch_tree(base: str) -> pd.DataFrame:
    """Page load-rows until it stops returning new players."""
    session = etl._make_session()
    headers = {**etl.DEFAULT_HEADERS,
               "Referer": f"https://www.draftsharks.com/{base}/{SLUG}",
               "HX-Request": "true"}
    rows, seen, offset = [], set(), 0
    while True:
        try:
            r = session.get(f"https://www.draftsharks.com/{base}/load-rows",
                            params={"offset": offset, "limit": PAGE,
                                    "pprSuperflexSlug": SLUG,
                                    "playerGroup": "all", "sort": ""},
                            headers=headers, timeout=45)
            r.raise_for_status()
        except Exception as e:
            raise RuntimeError(f"DraftSharks {base} offset={offset} failed: {e}") from e

        page = [x for x in parse_rows(r.text) if x["player_name"] not in seen]
        if not page:
            break
        seen.update(x["player_name"] for x in page)
        rows.extend(page)
        print(f"  {base:17} offset={offset:4} +{len(page):3} (total {len(rows)})")
        offset += PAGE
        time.sleep(0.5)   # be polite; this is someone else's server

    if len(rows) < MIN_ROWS:
        raise RuntimeError(
            f"DraftSharks {base}: only {len(rows)} rows (expected >= {MIN_ROWS}). "
            "Markup or endpoint likely changed -- check parse_rows before loading.")

    df = pd.DataFrame(rows)
    # Rank IS the published board order -- the page is pre-sorted, and load-rows
    # returns it in that order, so position in the sequence is the ranking.
    df["overall_rank"] = range(1, len(df) + 1)
    df["positional_rank"] = df.groupby("position_raw").cumcount() + 1
    return df


trees = {}
for base in TREES:
    trees[base] = fetch_tree(base)
    print(f"[ok] {base}: {len(trees[base])} rows\n")

raw_path = DATA / "raw" / f"draftsharks_two_tree_{etl.TODAY}.json"
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw_path.write_text(json.dumps({b: d.to_dict("records") for b, d in trees.items()},
                               indent=1), encoding="utf-8")
print(f"[ok] raw audit -> {raw_path}")

for base, d in trees.items():
    print(f"\n{base} by position:\n{d['position_raw'].value_counts().to_string()}")

  dynasty-rankings  offset=   0 +250 (total 250)


  dynasty-rankings  offset= 250 +250 (total 500)


  dynasty-rankings  offset= 500 +249 (total 749)


  dynasty-rankings  offset= 750 +150 (total 899)


[ok] dynasty-rankings: 899 rows



  rankings          offset=   0 +250 (total 250)


  rankings          offset= 250 +250 (total 500)


  rankings          offset= 500 +250 (total 750)


  rankings          offset= 750 +229 (total 979)


[ok] rankings: 979 rows

[ok] raw audit -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\raw\draftsharks_two_tree_2026-07-31.json

dynasty-rankings by position:
position_raw
WR    192
DB    162
DL    159
RB    120
TE     89
LB     89
QB     52
K      36

rankings by position:
position_raw
WR     206
DL     174
DB     157
RB     130
TE     108
LB      93
QB      42
K       37
DEF     32


In [3]:
# ---- Identity ---------------------------------------------------------------
def _slug(name: str) -> str:
    """Locally-minted player id. IDENTICAL rule to 04x_manual_dynasty_rankings,
    on purpose: it puts this automated pull in the same source_player_id
    namespace as the DynastySharks rows already resolved in the crosswalk. Using
    the vendor's numeric data-key instead would strand every one of them."""
    return re.sub(r"[^a-z0-9]+", "-", str(name).lower()).strip("-")


for d in trees.values():
    d["source_player_id"] = d["player_name"].map(_slug)
    d["source_name"] = SOURCE
    d["format"] = FORMAT
    d["source_uid"] = SOURCE + "|" + d["source_player_id"]

# Resolve over the UNION of both trees: redraft carries players dynasty omits
# (and vice versa), and one crosswalk row per source_player_id serves both.
identities = (pd.concat(trees.values(), ignore_index=True)
              .drop_duplicates("source_player_id")
              .assign(source=SOURCE, nfl_team=None)
              [["source", "source_player_id", "player_name", "position_raw", "nfl_team"]])
print(f"[info] {len(identities)} distinct players across both trees")

xwalk = etl.resolve_dynasty_crosswalk(identities, data_dir=str(DATA))
print(xwalk["match_method"].value_counts().to_string())

resolved = xwalk["gsis_id"].notna().mean()
print(f"[info] gsis resolved {xwalk['gsis_id'].notna().sum()}/{len(xwalk)} ({resolved:.1%})")

full = etl.upsert_dynasty_crosswalk(xwalk, DATA / "dim_dynasty_crosswalk.parquet")
print(f"[ok] crosswalk now {len(full)} rows across {full['source'].nunique()} sources")

[info] 1049 distinct players across both trees


match_method
exact             943
exact+disambig     55
unmatched          24
review             22
fuzzy               4
rookie              1
[info] gsis resolved 1002/1049 (95.5%)
[ok] crosswalk now 2174 rows across 3 sources


In [4]:
# ---- Transform to EAV + load ------------------------------------------------
gsis = dict(zip(xwalk["source_player_id"], xwalk["gsis_id"]))

folded = []
for base, prefix in TREES.items():
    d = trees[base]
    # prefix= is the whole point: both trees are source_name "DynastySharks",
    # so the SOURCE_PREFIX lookup would give them both "ds" and collide.
    long = etl.fold_ranks_long(d, prefix=prefix)
    # datetime64, matching the fact's existing schema -- etl.TODAY is an ISO
    # string, and mixing the two makes the column `object`, which arrow rejects.
    long["snapshot_date"] = pd.Timestamp(etl.TODAY)
    long["gsis_id"] = long["source_player_id"].map(gsis)
    folded.append(long)
    print(f"[ok] {base:17} -> {len(long)} EAV rows "
          f"({sorted(long['metric_key'].unique())})")

new = pd.concat(folded, ignore_index=True)[
    ["snapshot_date", "source_name", "source_player_id", "format", "source_uid",
     "gsis_id", "metric_key", "metric_num", "metric_text"]]

# Guard the model invariant before writing: each metric_key must belong to
# exactly one source_name across the WHOLE fact, not just this batch.
fact_path = DATA / "fact_dynasty_ranking_metrics.parquet"
existing = pd.read_parquet(fact_path, columns=["source_name", "metric_key"])
owners = (pd.concat([existing, new[["source_name", "metric_key"]]])
          .drop_duplicates().groupby("metric_key")["source_name"].nunique())
if (owners > 1).any():
    raise RuntimeError(f"metric_key owned by >1 source: {owners[owners > 1].index.tolist()}")

total = etl.load_replace_partition(new, fact_path,
                                   part_cols=("snapshot_date", "source_name"))
print(f"\n[ok] {len(new)} rows -> {fact_path} ({total} total)")

[ok] dynasty-rankings  -> 1798 EAV rows (['ds_overall_rank', 'ds_positional_rank'])
[ok] rankings          -> 1958 EAV rows (['dsr_overall_rank', 'dsr_positional_rank'])

[ok] 3756 rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\fact_dynasty_ranking_metrics.parquet (29764 total)


In [5]:
# ---- Verify -----------------------------------------------------------------
fact = pd.read_parquet(fact_path)
latest = fact[fact["snapshot_date"] == fact["snapshot_date"].max()]
print(latest.groupby(["source_name", "metric_key"]).size().to_string())

# The two trees must actually disagree -- if they rank identically, something is
# pulling the same board twice and the stance selector would be a no-op.
d = latest[latest["metric_key"] == "ds_overall_rank"].set_index("source_player_id")["metric_num"]
r = latest[latest["metric_key"] == "dsr_overall_rank"].set_index("source_player_id")["metric_num"]
both = pd.DataFrame({"dynasty": d, "redraft": r}).dropna()
# Pearson on the ranked columns *is* Spearman, and it avoids pulling scipy in
# (pandas method="spearman" imports scipy.stats) just for one assertion.
corr = both["dynasty"].rank().corr(both["redraft"].rank())
print(f"\n[info] {len(both)} players in both trees, rank spearman = {corr:.3f}")
assert corr < 0.99, "trees are effectively identical -- check TREES base paths"

print("\nbiggest dynasty-vs-redraft disagreements (redraft rank - dynasty rank):")
both["gap"] = both["redraft"] - both["dynasty"]
print(pd.concat([both.nlargest(5, "gap"), both.nsmallest(5, "gap")]).to_string())

source_name    metric_key         
DynastySharks  ds_overall_rank        899
               ds_positional_rank     899
               dsr_overall_rank       979
               dsr_positional_rank    979

[info] 829 players in both trees, rank spearman = 0.792

biggest dynasty-vs-redraft disagreements (redraft rank - dynasty rank):
                  dynasty  redraft    gap
source_player_id                         
ty-simpson           97.0    860.0  763.0
eli-stowers          89.0    751.0  662.0
j-j-mccarthy        166.0    824.0  658.0
drew-allar          255.0    893.0  638.0
david-bailey        240.0    855.0  615.0
trey-smack          885.0    233.0 -652.0
eddy-pineiro        887.0    236.0 -651.0
harrison-mevis      800.0    242.0 -558.0
jake-moody          888.0    352.0 -536.0
ryan-fitzgerald     886.0    371.0 -515.0
